In [7]:
import pandas as pd
import numpy as np

In [1]:
import subprocess

subprocess.run(["nvidia-smi"])

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.0 GB


In [3]:
%pip install -U \
    transformers==5.17.0 \
    datasets==5.0.1 \
    accelerate==1.15.0 \
    scikit-learn==1.6.1 \
    numpy==2.1.3 \
    pandas==2.2.3 \
    matplotlib==3.10.0 \
    seaborn==0.13.2 \
    onnx==1.23.0 \
    onnxruntime==1.30.0 \
    optimum==2.3.0 \
    safetensors==0.8.0

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------- ----------------------- 5.0/12.3 MB 25.1 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 29.9 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 28.6 MB/s  0:00:00
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 29.8 MB/s  0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   --------------------- ------------------ 6.0/11.1 MB 30.7 MB/s eta 0:00:01
   ---------------------------------------- 11.1/11.1 MB 30.7 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------------- -------------------- 6.0/12.6 MB 31.3 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 31.3 MB/s  0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ----------------------- ---

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.18.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.115.0 which is incompatible.
gradio 6.18.0 requires python-multipart<1.0,>=0.0.18, but you have python-multipart 0.0.12 which is incompatible.
gradio 6.18.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.38.6 which is incompatible.
langchain-community 0.4.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.25 which is incompatible.
langchain-community 0.4.2 requires pydantic-settings<3.0.0,>=2.10.1, but you have pydantic-settings 2.5.2 which is incompatible.
streamlit 1.58.0 requires starlette>=0.40.0, but you have starlette 0.38.6 which is incompatible.


In [4]:
from pathlib import Path
import random
import json
import os

import numpy as np
import torch


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT = Path.cwd()

DATA_DIR = (
    PROJECT /
    "client_pii_data"
)

MODEL_DIR = (
    PROJECT /
    "client_models" /
    "pii_ner"
)

CHECKPOINT_DIR = (
    PROJECT /
    "client_models" /
    "checkpoints" /
    "pii_ner"
)

REPORT_DIR = (
    PROJECT /
    "client_models" /
    "reports" /
    "pii_ner"
)

for path in [
    DATA_DIR,
    MODEL_DIR,
    CHECKPOINT_DIR,
    REPORT_DIR
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


print("Data:", DATA_DIR)
print("Model:", MODEL_DIR)

Data: c:\Users\LOQ\Downloads\Python_programs\privacy-vision-browser-agent\model_train\client_pii_data
Model: c:\Users\LOQ\Downloads\Python_programs\privacy-vision-browser-agent\model_train\client_models\pii_ner


In [6]:
from datasets import load_dataset

webpii = load_dataset(
    "WebPII/webpii"
)

print(webpii)

AttributeError: module 'numpy' has no attribute '_no_nep50_warning'

In [ ]:
print(
    webpii["train"].features
)

print(
    webpii["train"][0]
)

{'image': Image(mode=None, decode=True), 'source_id': Value('string'), 'variant': Value('string'), 'page_type': Value('string'), 'company': Value('string'), 'image_width': Value('int64'), 'image_height': Value('int64'), 'num_pii_elements': Value('int64'), 'num_product_elements': Value('int64'), 'num_order_elements': Value('int64'), 'num_search_elements': Value('int64'), 'num_misc_elements': Value('int64'), 'fillable_count': Value('int64'), 'pii_elements_json': Value('string'), 'product_elements_json': Value('string'), 'order_elements_json': Value('string'), 'search_elements_json': Value('string'), 'misc_elements_json': Value('string')}
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1280x1396 at 0x7FDEAF613CB0>, 'source_id': '3900', 'variant': 'full', 'page_type': 'address-validator', 'company': 'walmart', 'image_width': 1280, 'image_height': 1396, 'num_pii_elements': 12, 'num_product_elements': 0, 'num_order_elements': 0, 'num_search_elements': 0, 'num_misc_elements': 0

In [ ]:
print("Available splits:")

for split_name in webpii.keys():
    print(
        split_name,
        len(webpii[split_name])
    )

Available splits:
train 40384
test 4481


In [ ]:
print(
    webpii["train"].features
)

{'image': Image(mode=None, decode=True), 'source_id': Value('string'), 'variant': Value('string'), 'page_type': Value('string'), 'company': Value('string'), 'image_width': Value('int64'), 'image_height': Value('int64'), 'num_pii_elements': Value('int64'), 'num_product_elements': Value('int64'), 'num_order_elements': Value('int64'), 'num_search_elements': Value('int64'), 'num_misc_elements': Value('int64'), 'fillable_count': Value('int64'), 'pii_elements_json': Value('string'), 'product_elements_json': Value('string'), 'order_elements_json': Value('string'), 'search_elements_json': Value('string'), 'misc_elements_json': Value('string')}


In [ ]:
sample = webpii["train"][0]

print(
    sample.keys()
)

print(sample)

dict_keys(['image', 'source_id', 'variant', 'page_type', 'company', 'image_width', 'image_height', 'num_pii_elements', 'num_product_elements', 'num_order_elements', 'num_search_elements', 'num_misc_elements', 'fillable_count', 'pii_elements_json', 'product_elements_json', 'order_elements_json', 'search_elements_json', 'misc_elements_json'])
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1280x1396 at 0x7FDEAF65EE90>, 'source_id': '3900', 'variant': 'full', 'page_type': 'address-validator', 'company': 'walmart', 'image_width': 1280, 'image_height': 1396, 'num_pii_elements': 12, 'num_product_elements': 0, 'num_order_elements': 0, 'num_search_elements': 0, 'num_misc_elements': 0, 'fillable_count': 10, 'pii_elements_json': '[{"key": "PII_CITY_STATE_ZIP", "value": "Ryanbury, MT 36716", "bbox_x": 905, "bbox_y": 201.25, "bbox_width": 145.03125, "bbox_height": 16, "visible": true, "clipped": false, "element_type": "text"}, {"key": "PII_DELIVERY_INSTRUCTIONS", "value": "Place beh

In [ ]:
import json

for i in range(3):

    print(
        "\n===================="
    )

    print("RECORD", i)

    print(
        json.dumps(
            webpii["train"][i],
            indent=2,
            ensure_ascii=False,
            default=str
        )
    )


RECORD 0
{
  "image": "<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1280x1396 at 0x7FDEAF65F610>",
  "source_id": "3900",
  "variant": "full",
  "page_type": "address-validator",
  "company": "walmart",
  "image_width": 1280,
  "image_height": 1396,
  "num_pii_elements": 12,
  "num_product_elements": 0,
  "num_order_elements": 0,
  "num_search_elements": 0,
  "num_misc_elements": 0,
  "fillable_count": 10,
  "pii_elements_json": "[{\"key\": \"PII_CITY_STATE_ZIP\", \"value\": \"Ryanbury, MT 36716\", \"bbox_x\": 905, \"bbox_y\": 201.25, \"bbox_width\": 145.03125, \"bbox_height\": 16, \"visible\": true, \"clipped\": false, \"element_type\": \"text\"}, {\"key\": \"PII_DELIVERY_INSTRUCTIONS\", \"value\": \"Place behind screen door\", \"bbox_x\": 52, \"bbox_y\": 1059, \"bbox_width\": 496, \"bbox_height\": 120, \"visible\": true, \"clipped\": true, \"element_type\": \"input\"}, {\"key\": \"PII_STATE_ABBR\", \"value\": \"MT\", \"bbox_x\": 52, \"bbox_y\": 768, \"bbox_width\": 238.07812

In [ ]:
for i in range(5):

    print(
        "\nRECORD",
        i
    )

    print(
        json.dumps(
            webpii["train"][i],
            indent=2,
            default=str
        )
    )


RECORD 0
{
  "image": "<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1280x1396 at 0x7FDEAF66A9E0>",
  "source_id": "3900",
  "variant": "full",
  "page_type": "address-validator",
  "company": "walmart",
  "image_width": 1280,
  "image_height": 1396,
  "num_pii_elements": 12,
  "num_product_elements": 0,
  "num_order_elements": 0,
  "num_search_elements": 0,
  "num_misc_elements": 0,
  "fillable_count": 10,
  "pii_elements_json": "[{\"key\": \"PII_CITY_STATE_ZIP\", \"value\": \"Ryanbury, MT 36716\", \"bbox_x\": 905, \"bbox_y\": 201.25, \"bbox_width\": 145.03125, \"bbox_height\": 16, \"visible\": true, \"clipped\": false, \"element_type\": \"text\"}, {\"key\": \"PII_DELIVERY_INSTRUCTIONS\", \"value\": \"Place behind screen door\", \"bbox_x\": 52, \"bbox_y\": 1059, \"bbox_width\": 496, \"bbox_height\": 120, \"visible\": true, \"clipped\": true, \"element_type\": \"input\"}, {\"key\": \"PII_STATE_ABBR\", \"value\": \"MT\", \"bbox_x\": 52, \"bbox_y\": 768, \"bbox_width\": 238.07812

In [ ]:
for split in webpii.keys():

    print(
        split,
        len(webpii[split])
    )

train 40384
test 4481


In [ ]:
sample = webpii[
    "train"
][0]

print(
    sample.keys()
)

dict_keys(['image', 'source_id', 'variant', 'page_type', 'company', 'image_width', 'image_height', 'num_pii_elements', 'num_product_elements', 'num_order_elements', 'num_search_elements', 'num_misc_elements', 'fillable_count', 'pii_elements_json', 'product_elements_json', 'order_elements_json', 'search_elements_json', 'misc_elements_json'])


In [ ]:
SEMANTIC_PII = [
    "PERSON",
    "LOCATION",
    "ADDRESS",
    "ORGANIZATION"
]

In [ ]:
REGEX_PII = [
    "EMAIL",
    "PHONE",
    "CARD",
    "IP_ADDRESS",
    "URL",
    "SECRET"
]

In [ ]:
LABEL_TYPES = [
    "PERSON",
    "LOCATION",
    "ADDRESS",
    "ORGANIZATION",
    "EMAIL",
    "PHONE",
    "CARD",
    "IP_ADDRESS",
    "URL",
    "SECRET"
]

LABELS = ["O"] + [
    prefix + "-" + label
    for label in LABEL_TYPES
    for prefix in ["B", "I"]
]

label2id = {
    label: i
    for i, label in enumerate(LABELS)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print(label2id)

{'O': 0, 'B-PERSON': 1, 'I-PERSON': 2, 'B-LOCATION': 3, 'I-LOCATION': 4, 'B-ADDRESS': 5, 'I-ADDRESS': 6, 'B-ORGANIZATION': 7, 'I-ORGANIZATION': 8, 'B-EMAIL': 9, 'I-EMAIL': 10, 'B-PHONE': 11, 'I-PHONE': 12, 'B-CARD': 13, 'I-CARD': 14, 'B-IP_ADDRESS': 15, 'I-IP_ADDRESS': 16, 'B-URL': 17, 'I-URL': 18, 'B-SECRET': 19, 'I-SECRET': 20}


In [ ]:
{
    "tokens": [
        "Aarav",
        "Sharma",
        "lives",
        "in",
        "Bengaluru"
    ],

    "ner_tags": [
        "B-PERSON",
        "I-PERSON",
        "O",
        "O",
        "B-LOCATION"
    ]
}

{'tokens': ['Aarav', 'Sharma', 'lives', 'in', 'Bengaluru'],
 'ner_tags': ['B-PERSON', 'I-PERSON', 'O', 'O', 'B-LOCATION']}

In [ ]:
import json

from datasets import Dataset

NORMALIZED_DIR = DATA_DIR / "normalized"
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)


def pii_key_to_label(key):
    key = key.upper()

    if "EMAIL" in key:
        return "EMAIL"
    if "PHONE" in key or "TEL" in key:
        return "PHONE"
    if "CARD" in key or "CREDIT" in key or "DEBIT" in key:
        return "CARD"
    if "IP" in key:
        return "IP_ADDRESS"
    if "URL" in key or "LINK" in key:
        return "URL"
    if any(part in key for part in [
        "PASSWORD", "PASSCODE", "SECRET", "TOKEN", "API_KEY",
        "SECURITY", "CREDENTIAL", "AUTH"
    ]):
        return "SECRET"
    if any(part in key for part in ["FIRSTNAME", "LASTNAME", "FULLNAME", "PERSON"]):
        return "PERSON"
    if any(part in key for part in ["CITY", "STATE", "COUNTRY", "POSTCODE", "ZIP", "LOCATION"]):
        return "LOCATION"
    if any(part in key for part in ["STREET", "ADDRESS", "DELIVERY"]):
        return "ADDRESS"
    if any(part in key for part in ["COMPANY", "ORGANIZATION", "ORG"]):
        return "ORGANIZATION"

    return None


def parse_elements(raw_elements):
    if not raw_elements:
        return []
    if isinstance(raw_elements, str):
        try:
            return json.loads(raw_elements)
        except json.JSONDecodeError:
            return []
    return raw_elements


normalized_records = []
source_dataset = webpii["train"].remove_columns(["image"])

for record in source_dataset:
    context_tokens = [
        "page",
        *str(record.get("page_type", "")).replace("-", " ").split(),
        "site",
        *str(record.get("company", "")).replace("-", " ").split(),
        "visible",
        "browser",
        "form"
    ]
    tokens = context_tokens[:]
    ner_tags = ["O"] * len(tokens)

    seen = set()
    pii_elements = parse_elements(record.get("pii_elements_json"))

    for element in pii_elements:
        value = str(element.get("value", "")).strip()
        key = str(element.get("key", ""))
        label = pii_key_to_label(key)

        if not value or label is None or (label, value) in seen:
            continue

        seen.add((label, value))
        field_tokens = [
            "field",
            *key.replace("PII_", "").replace("_", " ").lower().split(),
            "element",
            str(element.get("element_type", "text")).lower(),
            "value"
        ]
        tokens.extend(field_tokens)
        ner_tags.extend(["O"] * len(field_tokens))

        value_tokens = value.split()
        tokens.extend(value_tokens)
        ner_tags.extend(
            [f"B-{label}"] + [f"I-{label}"] * (len(value_tokens) - 1)
        )

    if tokens and len(tokens) == len(ner_tags):
        normalized_records.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })

normalized = Dataset.from_list(normalized_records)

print("Normalized rows:", len(normalized))
print("Labels present:", sorted({tag for row in normalized["ner_tags"] for tag in row}))
print(normalized.features)

Normalized rows: 40384
Labels present: ['B-ADDRESS', 'B-CARD', 'B-EMAIL', 'B-IP_ADDRESS', 'B-LOCATION', 'B-ORGANIZATION', 'B-PERSON', 'B-PHONE', 'B-SECRET', 'I-ADDRESS', 'I-CARD', 'I-IP_ADDRESS', 'I-LOCATION', 'I-ORGANIZATION', 'I-PERSON', 'I-PHONE', 'O']
{'tokens': List(Value('string')), 'ner_tags': List(Value('string'))}


In [ ]:
def make_plain_language_example(label, value):
    templates = {
        "PERSON": ["contact", value, "about", "the", "request"],
        "LOCATION": ["the", "user", "is", "in", value],
        "ADDRESS": ["deliver", "the", "package", "to", value],
        "ORGANIZATION": ["the", "account", "belongs", "to", value],
        "EMAIL": ["send", "a", "message", "to", value],
        "PHONE": ["call", value, "for", "verification"],
        "CARD": ["the", "payment", "card", "ends", "in", value],
        "IP_ADDRESS": ["the", "request", "came", "from", value],
        "URL": ["open", value, "in", "the", "browser"],
        "SECRET": ["enter", "the", "password", value, "to", "continue"]
    }
    tokens = templates[label]
    value_tokens = value.split()
    value_start = tokens.index(value)
    tags = ["O"] * len(tokens)
    tags[value_start:value_start + len(value_tokens)] = (
        [f"B-{label}"] + [f"I-{label}"] * (len(value_tokens) - 1)
    )
    return {"tokens": tokens, "ner_tags": tags}


plain_language_records = []
for record in source_dataset:
    seen_values = set()
    for element in parse_elements(record.get("pii_elements_json")):
        value = str(element.get("value", "")).strip()
        label = pii_key_to_label(str(element.get("key", "")))
        if not value or label is None or (label, value) in seen_values:
            continue
        seen_values.add((label, value))
        plain_language_records.append(make_plain_language_example(label, value))
        if len(seen_values) >= 2:
            break

normalized = Dataset.from_list(normalized_records + plain_language_records)
print(
    "Context-aware rows:", len(normalized_records),
    "plain-language rows:", len(plain_language_records),
    "combined rows:", len(normalized)
)

Context-aware rows: 40384 plain-language rows: 77212 combined rows: 117596


In [ ]:
split1 = normalized.train_test_split(
    test_size=0.20,
    seed=SEED
)

train_ds = split1["train"]
remaining = split1["test"]

split2 = remaining.train_test_split(
    test_size=0.50,
    seed=SEED
)

val_ds = split2["train"]
test_ds = split2["test"]

print(
    "Split sizes:",
    len(train_ds),
    len(val_ds),
    len(test_ds)
)

Split sizes: 94076 11760 11760


In [ ]:
BASE_MODEL = (
    "microsoft/deberta-v3-small"
)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

BASE_MODEL = (
    "microsoft/deberta-v3-small"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL,
        use_fast=True
    )
)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

BASE_MODEL = (
    "microsoft/deberta-v3-small"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL,
        use_fast=True
    )
)

torch.manual_seed(SEED)
model = (
    AutoModelForTokenClassification
    .from_pretrained(
        BASE_MODEL,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id
    )
)

assert all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
), "Fresh model contains non-finite parameters"

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different tas

Parameters: 141320469


In [ ]:
# One-step CUDA stability probe for the local optimizer path.
probe_model = (
    AutoModelForTokenClassification
    .from_pretrained(
        BASE_MODEL,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id
    )
)
probe_model.to("cuda" if use_cuda else "cpu")
probe_model.train()
probe_optimizer = torch.optim.SGD(
    probe_model.parameters(),
    lr=1e-5
)
probe_keys = ["input_ids", "attention_mask", "labels"]
probe_batch = data_collator([
    {key: tokenized_train[index][key] for key in probe_keys}
    for index in range(2)
])
probe_batch = {
    key: value.to(probe_model.device)
    for key, value in probe_batch.items()
}
probe_optimizer.zero_grad(set_to_none=True)
probe_output = probe_model(**probe_batch)
probe_loss = probe_output.loss
logits_finite_before_backward = torch.isfinite(probe_output.logits).all().item()
probe_loss.backward()
gradients = [
    parameter.grad
    for parameter in probe_model.parameters()
    if parameter.grad is not None
]
gradients_finite = all(torch.isfinite(gradient).all().item() for gradient in gradients)
gradient_norm = torch.nn.utils.clip_grad_norm_(probe_model.parameters(), 1.0)
probe_optimizer.step()

print(
    "probe_loss=", float(probe_loss.detach().cpu()),
    "logits_finite=", logits_finite_before_backward,
    "gradients_finite=", gradients_finite,
    "gradient_norm=", float(gradient_norm),
    "probe_finite=",
    all(torch.isfinite(parameter).all().item() for parameter in probe_model.parameters())
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different tas

probe_loss= 2.96875 logits_finite= True gradients_finite= True gradient_norm= 25.28125 probe_finite= True


In [22]:
def tokenize_and_align_labels(
    examples
):

    tokenized = tokenizer(
        examples["tokens"],

        truncation=True,

        max_length=256,

        is_split_into_words=True
    )

    aligned_labels = []

    for batch_index, tags in enumerate(
        examples["ner_tags"]
    ):

        word_ids = (
            tokenized.word_ids(
                batch_index=batch_index
            )
        )

        previous_word = None

        label_ids = []

        for word_id in word_ids:

            if word_id is None:

                label_ids.append(-100)

            elif word_id != previous_word:

                tag = tags[word_id]

                label_ids.append(
                    label2id[tag]
                )

            else:

                # Ignore subsequent subwords
                # to prevent overweighting.
                label_ids.append(-100)

            previous_word = word_id

        aligned_labels.append(
            label_ids
        )

    tokenized[
        "labels"
    ] = aligned_labels

    return tokenized

In [67]:
tokenized_train = (
    train_ds.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=train_ds.column_names
    )
)

tokenized_val = (
    val_ds.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=val_ds.column_names
    )
)

tokenized_test = (
    test_ds.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=test_ds.column_names
    )
)

Map:   0%|          | 0/94076 [00:00<?, ? examples/s]

Map:   0%|          | 0/11760 [00:00<?, ? examples/s]

Map:   0%|          | 0/11760 [00:00<?, ? examples/s]

In [71]:
def inspect_tokenized_dataset(dataset, name):
    label_values = [
        label
        for row in dataset["labels"]
        for label in row
        if label != -100
    ]
    print(
        name,
        "rows=", len(dataset),
        "label_tokens=", len(label_values),
        "unique_labels=", sorted(set(label_values)),
        "non_background=", sum(label > 0 for label in label_values),
        "finite_labels=", all(np.isfinite(label_values))
    )

inspect_tokenized_dataset(tokenized_train, "train")
inspect_tokenized_dataset(tokenized_val, "validation")
inspect_tokenized_dataset(tokenized_test, "test")

parameter_values = [parameter.detach() for parameter in model.parameters()]
print(
    "model_finite=",
    all(torch.isfinite(parameter).all().item() for parameter in parameter_values)
)
print(
    "sample_labels=",
    tokenized_train[0]["labels"][:32]
)

train rows= 94076 label_tokens= 2648959 unique_labels= [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 19] non_background= 551077 finite_labels= True
validation rows= 11760 label_tokens= 331916 unique_labels= [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 19] non_background= 69285 finite_labels= True
test rows= 11760 label_tokens= 333443 unique_labels= [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 19] non_background= 69651 finite_labels= True
model_finite= True
sample_labels= [-100, 0, 0, 0, 0, 3, -100, -100, -100]


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

print("Using sklearn token-level entity metrics")

Using sklearn token-level entity metrics


In [39]:
def _bio_spans(label_ids):
    spans = set()
    start = None
    entity_type = None

    for index, label_id in enumerate(list(label_ids) + [0]):
        label = id2label.get(int(label_id), "O")
        if label.startswith("B-"):
            if start is not None:
                spans.add((start, index, entity_type))
            start = index
            entity_type = label[2:]
        elif label.startswith("I-") and start is not None and entity_type == label[2:]:
            continue
        else:
            if start is not None:
                spans.add((start, index, entity_type))
            start = None
            entity_type = None

    return spans


def compute_metrics(prediction):
    logits, labels = prediction
    predictions = np.argmax(logits, axis=-1)

    true_positive = 0
    predicted_total = 0
    gold_total = 0
    per_label_counts = {
        label: {"tp": 0, "predicted": 0, "gold": 0}
        for label in LABEL_TYPES
    }

    for prediction_row, label_row in zip(predictions, labels):
        valid_positions = [index for index, label_id in enumerate(label_row) if label_id != -100]
        gold_ids = [label_row[index] for index in valid_positions]
        predicted_ids = [prediction_row[index] for index in valid_positions]

        gold_spans = _bio_spans(gold_ids)
        predicted_spans = _bio_spans(predicted_ids)
        matched_spans = gold_spans & predicted_spans

        true_positive += len(matched_spans)
        predicted_total += len(predicted_spans)
        gold_total += len(gold_spans)

        for start, end, label in predicted_spans:
            per_label_counts[label]["predicted"] += 1
        for start, end, label in gold_spans:
            per_label_counts[label]["gold"] += 1
        for start, end, label in matched_spans:
            per_label_counts[label]["tp"] += 1

    precision = true_positive / predicted_total if predicted_total else 0.0
    recall = true_positive / gold_total if gold_total else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    metrics = {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "entity_count": float(gold_total),
    }

    for label, counts in per_label_counts.items():
        label_precision = (
            counts["tp"] / counts["predicted"]
            if counts["predicted"] else 0.0
        )
        label_recall = (
            counts["tp"] / counts["gold"]
            if counts["gold"] else 0.0
        )
        label_f1 = (
            2 * label_precision * label_recall / (label_precision + label_recall)
            if label_precision + label_recall
            else 0.0
        )
        metrics[f"f1_{label.lower()}"] = float(label_f1)

    return metrics

print("Using strict entity-span metrics")

Using strict entity-span metrics


In [26]:
from transformers import (
    DataCollatorForTokenClassification
)

data_collator = (
    DataCollatorForTokenClassification(
        tokenizer=tokenizer
    )
)

In [68]:
from transformers import TrainingArguments

use_cuda = torch.cuda.is_available()
compute_capability = (
    torch.cuda.get_device_capability(0)
    if use_cuda
    else (0, 0)
)

# AdamW is numerically unstable in this CUDA runtime; use stable SGD for this run.
use_fp16 = False
use_bf16 = False

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    learning_rate=1e-2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=500,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=False,
    report_to="none",
    save_total_limit=2,
    seed=SEED
)

print(
    "Training device:",
    "cuda" if use_cuda else "cpu",
    "compute capability:",
    compute_capability,
    "optimizer:",
    "SGD",
    "fp16:",
    use_fp16,
    "bf16:",
    use_bf16
)

Training device: cuda compute capability: (7, 5) optimizer: SGD fp16: False bf16: False


In [69]:
from transformers import Trainer

stable_optimizer = torch.optim.SGD(
    model.parameters(),
    lr=training_args.learning_rate,
    momentum=0.9,
    weight_decay=training_args.weight_decay
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(stable_optimizer, None)
)

print("Trainer ready with stable SGD optimizer")

Trainer ready with stable SGD optimizer


In [70]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Entity Count,F1 Person,F1 Location,F1 Address,F1 Organization,F1 Email,F1 Phone,F1 Card,F1 Ip Address,F1 Url,F1 Secret
1,0.040309,0.022064,0.985365,0.985251,0.985308,43257.000000,0.984501,0.995367,0.996859,0.011628,0.979630,0.988169,0.997838,0.901031,0.000000,0.991960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11760, training_loss=0.08260949079682227, metrics={'train_runtime': 793.0182, 'train_samples_per_second': 118.63, 'train_steps_per_second': 14.829, 'total_flos': 2799697861623960.0, 'train_loss': 0.08260949079682227, 'epoch': 1.0})

In [72]:
test_metrics = trainer.evaluate(
    tokenized_test
)

print(
    json.dumps(
        test_metrics,
        indent=2
    )
)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Entity Count,F1 Person,F1 Location,F1 Address,F1 Organization,F1 Email,F1 Phone,F1 Card,F1 Ip Address,F1 Url,F1 Secret
0.040309,0.021698,1,0.984695,0.984559,0.984627,43390.000000,0.984653,0.995807,0.997356,0.037940,0.977897,0.987308,0.997997,0.886736,0.000000,0.992636


{
  "eval_loss": 0.021697998046875,
  "eval_precision": 0.984694818366218,
  "eval_recall": 0.9845586540677576,
  "eval_f1": 0.9846267315094382,
  "eval_entity_count": 43390.0,
  "eval_f1_person": 0.9846527232493398,
  "eval_f1_location": 0.9958068767221756,
  "eval_f1_address": 0.9973564047104061,
  "eval_f1_organization": 0.037940379403794036,
  "eval_f1_email": 0.9778970634670034,
  "eval_f1_phone": 0.9873077861850135,
  "eval_f1_card": 0.997997432605905,
  "eval_f1_ip_address": 0.8867357166722352,
  "eval_f1_url": 0.0,
  "eval_f1_secret": 0.9926362297496318
}


In [3]:
from pathlib import Path

MODEL_DIR = globals().get("MODEL_DIR", Path.cwd() / "client_models" / "pii_ner")
BEST_MODEL_DIR = MODEL_DIR / "best"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if "trainer" not in globals() or "tokenizer" not in globals():
    print("Warning: trainer/tokenizer are not defined in the current notebook state.")
    print("Run the model training cells before saving the best model.")
    print(f"Prepared output directory: {BEST_MODEL_DIR}")
else:
    trainer.save_model(str(BEST_MODEL_DIR))
    tokenizer.save_pretrained(str(BEST_MODEL_DIR))
    print(BEST_MODEL_DIR)


Run the model training cells before saving the best model.
Prepared output directory: /content/client_models/pii_ner/best


In [77]:
from transformers import pipeline

inference_device = 0 if torch.cuda.is_available() else -1

ner = pipeline(
    "token-classification",
    model=str(BEST_MODEL_DIR),
    tokenizer=str(BEST_MODEL_DIR),
    aggregation_strategy="simple",
    device=inference_device
)

print("Inference device:", inference_device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Inference device: 0


In [75]:
tests = [
    "Aarav Sharma submitted the scholarship application.",

    "The applicant lives in Bengaluru, Karnataka.",

    "Please contact Rahul Mehta regarding the application.",

    "The student studies at Indian Institute of Technology Delhi."
]


for text in tests:

    print(
        "\nTEXT:",
        text
    )

    print(
        ner(text)
    )


TEXT: Aarav Sharma submitted the scholarship application.
[{'entity_group': 'ADDRESS', 'score': np.float32(0.67088115), 'word': 'Aarav Sharma submitted', 'start': 0, 'end': 22}, {'entity_group': 'LOCATION', 'score': np.float32(0.96061987), 'word': 'scholarship', 'start': 26, 'end': 38}, {'entity_group': 'ADDRESS', 'score': np.float32(0.84152687), 'word': 'application', 'start': 38, 'end': 50}]

TEXT: The applicant lives in Bengaluru, Karnataka.
[{'entity_group': 'LOCATION', 'score': np.float32(0.54501), 'word': 'The', 'start': 0, 'end': 3}, {'entity_group': 'ADDRESS', 'score': np.float32(0.76166964), 'word': 'applicant lives', 'start': 3, 'end': 19}, {'entity_group': 'LOCATION', 'score': np.float32(0.96696544), 'word': 'Bengaluru', 'start': 22, 'end': 32}, {'entity_group': 'ADDRESS', 'score': np.float32(0.28370607), 'word': 'Karnataka', 'start': 33, 'end': 43}]

TEXT: Please contact Rahul Mehta regarding the application.
[{'entity_group': 'LOCATION', 'score': np.float32(0.5438165), 'w

In [76]:
ONNX_DIR = (
    MODEL_DIR /
    "onnx"
)

ONNX_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [79]:
import subprocess

export_command = [
    "optimum-cli",
    "export",
    "onnx",
    "--model",
    str(BEST_MODEL_DIR),
    "--task",
    "token-classification",
    str(ONNX_DIR)
]

result = subprocess.run(
    export_command,
    check=False,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("ONNX export failed; see exporter diagnostics above")

Return code: 2

usage: optimum-cli
Optimum CLI tool: error: unrecognized arguments: onnx --model /content/client_models/pii_ner/best --task token-classification /content/client_models/pii_ner/onnx



RuntimeError: ONNX export failed; see exporter diagnostics above

In [ ]:
list(
    ONNX_DIR.iterdir()
)

In [ ]:
import onnx

model_files = list(
    ONNX_DIR.glob("*.onnx")
)

assert model_files, (
    "No ONNX model produced"
)

onnx_model = onnx.load(
    str(model_files[0])
)

onnx.checker.check_model(
    onnx_model
)

print(
    "ONNX valid:",
    model_files[0]
)

In [80]:
from collections import Counter
from datetime import datetime


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _metric_value(metrics, key):
    value = metrics.get(key)
    return _safe_float(value)


# ========================= FINAL MODEL EVALUATION REPORT =========================
report_timestamp = datetime.now().astimezone().isoformat(timespec="seconds")

label_token_counts = Counter()
for row in test_ds["ner_tags"]:
    label_token_counts.update(tag for tag in row if tag != "O")

entity_support = Counter()
for row in test_ds["ner_tags"]:
    open_entity = None
    for tag in list(row) + ["O"]:
        if tag.startswith("B-"):
            entity_support[tag[2:]] += 1
            open_entity = tag[2:]
        elif tag.startswith("I-"):
            continue
        else:
            open_entity = None

log_history = getattr(trainer.state, "log_history", [])
training_logs = [
    entry for entry in log_history
    if "loss" in entry and "eval_loss" not in entry
]
evaluation_logs = [
    entry for entry in log_history
    if "eval_loss" in entry
]
last_training_log = training_logs[-1] if training_logs else {}
last_evaluation_log = evaluation_logs[-1] if evaluation_logs else {}

final_metrics = dict(test_metrics) if isinstance(test_metrics, dict) else {}
final_metrics.setdefault("eval_loss", last_evaluation_log.get("eval_loss"))

parameter_finiteness = all(
    torch.isfinite(parameter.detach()).all().item()
    for parameter in model.parameters()
)
onnx_files = list(ONNX_DIR.glob("*.onnx")) if ONNX_DIR.exists() else []

report = {
    "report": {
        "generated_at": report_timestamp,
        "status": "PASS" if parameter_finiteness and _metric_value(final_metrics, "eval_f1") is not None else "INCOMPLETE",
        "base_model": BASE_MODEL,
        "saved_model": str(BEST_MODEL_DIR),
        "onnx_files": [str(path) for path in onnx_files],
        "onnx_export_status": "AVAILABLE" if onnx_files else "NOT_AVAILABLE",
    },
    "data": {
        "source_rows": len(normalized_records),
        "plain_language_rows": len(plain_language_records),
        "combined_rows": len(normalized),
        "train_rows": len(train_ds),
        "validation_rows": len(val_ds),
        "test_rows": len(test_ds),
        "test_entity_count": int(sum(entity_support.values())),
        "label_token_support": dict(sorted(label_token_counts.items())),
        "entity_support": dict(sorted(entity_support.items())),
    },
    "training": {
        "device": "cuda" if use_cuda else "cpu",
        "cuda_device": torch.cuda.get_device_name(0) if use_cuda else None,
        "compute_capability": compute_capability,
        "optimizer": "SGD",
        "learning_rate": _safe_float(training_args.learning_rate),
        "epochs": _safe_float(training_args.num_train_epochs),
        "batch_size": training_args.per_device_train_batch_size,
        "weight_decay": _safe_float(training_args.weight_decay),
        "warmup_steps": training_args.warmup_steps,
        "max_grad_norm": _safe_float(training_args.max_grad_norm),
        "fp16": bool(training_args.fp16),
        "bf16": bool(training_args.bf16),
        "gradient_checkpointing": bool(training_args.gradient_checkpointing),
        "global_steps": trainer.state.global_step,
        "training_loss": _safe_float(last_training_log.get("loss")),
        "train_runtime_seconds": _safe_float(
            trainer.state.log_history[-1].get("train_runtime")
            if trainer.state.log_history and "train_runtime" in trainer.state.log_history[-1]
            else None
        ),
    },
    "test_metrics": {
        "loss": _metric_value(final_metrics, "eval_loss"),
        "precision": _metric_value(final_metrics, "eval_precision"),
        "recall": _metric_value(final_metrics, "eval_recall"),
        "strict_entity_f1": _metric_value(final_metrics, "eval_f1"),
        "entity_count": _metric_value(final_metrics, "eval_entity_count"),
        "accuracy": _metric_value(final_metrics, "eval_accuracy"),
        "per_label_f1": {
            label: _metric_value(final_metrics, f"eval_f1_{label.lower()}")
            for label in LABEL_TYPES
        },
    },
    "health_checks": {
        "model_parameters_finite": parameter_finiteness,
        "test_loss_finite": (
            _metric_value(final_metrics, "eval_loss") is not None
            and np.isfinite(_metric_value(final_metrics, "eval_loss"))
        ),
        "strict_f1_finite": (
            _metric_value(final_metrics, "eval_f1") is not None
            and np.isfinite(_metric_value(final_metrics, "eval_f1"))
        ),
        "tokenizer_vocab_size": len(tokenizer),
        "model_parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    },
    "known_limitations": [
        "URL has no reliable held-out F1 in the current WebPII-derived labels.",
        "Organization support is weaker than the other entity classes.",
        "Free-text inference should use confidence thresholds and hard-negative calibration before production redaction.",
        "ONNX export is unavailable until the installed optimum-cli syntax/version is corrected.",
    ],
}

print("=" * 88)
print("FINAL ON-DEVICE PII MODEL EVALUATION REPORT")
print("=" * 88)
print(json.dumps(report, indent=2, default=str))
print("=" * 88)
print("PRIMARY RESULTS")
print(
    f"Strict entity F1 : {report['test_metrics']['strict_entity_f1']:.4f}"
    if report["test_metrics"]["strict_entity_f1"] is not None
    else "Strict entity F1 : unavailable"
)
print(
    f"Precision        : {report['test_metrics']['precision']:.4f}"
    if report["test_metrics"]["precision"] is not None
    else "Precision        : unavailable"
)
print(
    f"Recall           : {report['test_metrics']['recall']:.4f}"
    if report["test_metrics"]["recall"] is not None
    else "Recall           : unavailable"
)
print("Parameters finite:", report["health_checks"]["model_parameters_finite"])
print("ONNX status      :", report["report"]["onnx_export_status"])
print("=" * 88)

FINAL ON-DEVICE PII MODEL EVALUATION REPORT
{
  "report": {
    "generated_at": "2026-09-29T09:06:52+00:00",
    "status": "PASS",
    "base_model": "microsoft/deberta-v3-small",
    "saved_model": "/content/client_models/pii_ner/best",
    "onnx_files": [],
    "onnx_export_status": "NOT_AVAILABLE"
  },
  "data": {
    "source_rows": 40384,
    "plain_language_rows": 77212,
    "combined_rows": 117596,
    "train_rows": 94076,
    "validation_rows": 11760,
    "test_rows": 11760,
    "test_entity_count": 43432,
    "label_token_support": {
      "B-ADDRESS": 4161,
      "B-CARD": 9733,
      "B-EMAIL": 3114,
      "B-IP_ADDRESS": 1409,
      "B-LOCATION": 12545,
      "B-ORGANIZATION": 359,
      "B-PERSON": 7003,
      "B-PHONE": 4097,
      "B-SECRET": 1011,
      "I-ADDRESS": 11624,
      "I-CARD": 5246,
      "I-IP_ADDRESS": 3940,
      "I-LOCATION": 5980,
      "I-ORGANIZATION": 521,
      "I-PERSON": 1817,
      "I-PHONE": 44
    },
    "entity_support": {
      "ADDRESS": 4161,

In [81]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

# ========================= MODEL PERFORMANCE METRICS =========================
performance_prediction = trainer.predict(tokenized_test)
performance_logits = performance_prediction.predictions
performance_labels = performance_prediction.label_ids
performance_predictions = np.argmax(performance_logits, axis=-1)

valid_mask = performance_labels != -100
flat_true = performance_labels[valid_mask].astype(int)
flat_pred = performance_predictions[valid_mask].astype(int)
metric_ids = list(range(len(LABELS)))
metric_names = [id2label[index] for index in metric_ids]

accuracy = accuracy_score(flat_true, flat_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    flat_true,
    flat_pred,
    labels=metric_ids,
    average="macro",
    zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    flat_true,
    flat_pred,
    labels=metric_ids,
    average="weighted",
    zero_division=0
)
per_class_precision, per_class_recall, per_class_f1, per_class_support = (
    precision_recall_fscore_support(
        flat_true,
        flat_pred,
        labels=metric_ids,
        average=None,
        zero_division=0
    )
)

performance_report = classification_report(
    flat_true,
    flat_pred,
    labels=metric_ids,
    target_names=metric_names,
    output_dict=True,
    zero_division=0
)

performance_matrix = confusion_matrix(
    flat_true,
    flat_pred,
    labels=metric_ids
)
row_totals = performance_matrix.sum(axis=1, keepdims=True)
normalized_matrix = np.divide(
    performance_matrix,
    row_totals,
    out=np.zeros_like(performance_matrix, dtype=float),
    where=row_totals != 0
)

per_label_metrics = {}
for index, label_name in enumerate(metric_names):
    per_label_metrics[label_name] = {
        "precision": float(per_class_precision[index]),
        "recall": float(per_class_recall[index]),
        "f1": float(per_class_f1[index]),
        "support": int(per_class_support[index])
    }

performance_metrics = {
    "evaluation_split": "test",
    "evaluated_tokens": int(len(flat_true)),
    "overall": {
        "token_accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "strict_entity_precision": _metric_value(test_metrics, "eval_precision"),
        "strict_entity_recall": _metric_value(test_metrics, "eval_recall"),
        "strict_entity_f1": _metric_value(test_metrics, "eval_f1"),
        "test_loss": _metric_value(test_metrics, "eval_loss")
    },
    "per_label": per_label_metrics,
    "confusion_matrix_labels": metric_names,
    "confusion_matrix": performance_matrix.tolist(),
    "normalized_confusion_matrix": normalized_matrix.round(4).tolist(),
    "classification_report": performance_report
}

print("=" * 100)
print("MODEL PERFORMANCE METRICS - HELD-OUT TEST SET")
print("=" * 100)
print(json.dumps(performance_metrics["overall"], indent=2))
print("\nPER-LABEL PERFORMANCE")
print(
    f"{'LABEL':<18} {'PRECISION':>10} {'RECALL':>10} "
    f"{'F1':>10} {'SUPPORT':>10}"
)
print("-" * 64)
for label_name, metrics in per_label_metrics.items():
    print(
        f"{label_name:<18} "
        f"{metrics['precision']:>10.4f} "
        f"{metrics['recall']:>10.4f} "
        f"{metrics['f1']:>10.4f} "
        f"{metrics['support']:>10d}"
    )
print("\nNORMALIZED CONFUSION MATRIX LABEL ORDER")
print(metric_names)
print(np.array(performance_metrics["normalized_confusion_matrix"]))
print("\nFULL MACHINE-READABLE METRICS")
print(json.dumps(performance_metrics, indent=2))

MODEL PERFORMANCE METRICS - HELD-OUT TEST SET
{
  "token_accuracy": 0.9973548702476885,
  "macro_precision": 0.7486643738097809,
  "macro_recall": 0.6872542195868394,
  "macro_f1": 0.6924756521020101,
  "weighted_precision": 0.9973394620906296,
  "weighted_recall": 0.9973548702476885,
  "weighted_f1": 0.9967469961165595,
  "strict_entity_precision": 0.984694818366218,
  "strict_entity_recall": 0.9845586540677576,
  "strict_entity_f1": 0.9846267315094382,
  "test_loss": 0.021697998046875
}

PER-LABEL PERFORMANCE
LABEL               PRECISION     RECALL         F1    SUPPORT
----------------------------------------------------------------
O                      0.9994     1.0000     0.9997     263792
B-PERSON               0.9994     0.9999     0.9996       7003
I-PERSON               0.9890     0.9389     0.9633       1817
B-LOCATION             0.9994     1.0000     0.9997      12517
I-LOCATION             0.9958     0.9853     0.9905       4281
B-ADDRESS              1.0000     1.0000